In [2]:
import numpy as np
import scipy.linalg as la
import random
import math
import csv
from random import randint
import matplotlib as mpl
import matplotlib.pyplot as plt
import pygad


# Modelo de Hamiltoniano de Heisenberg independiente del tiempo utilizando algoritmo genético.

Dada la estabilidad de los resultados para Hxx, en lugar de pasar directamente al hamiltoniano de Heisenberg lo hacemos paulatinamente apartandonos del sistema a través de una constante $\Delta \in [0,1]$
$\begin{equation*}
H\ = \sum _{i=1}^{N} J_i \left( \sigma _{i}^{x} \sigma _{i+1}^{x} +\sigma _{i}^{y} \sigma _{i+1}^{y} + \Delta \sigma _{i}^{z} \sigma _{i+1}^{z}\right )
\end{equation*}$

generamos una población de espines aleatoria.

Buscamos encontrar parametros que maximicen la fidelidad en la transmisión utilizando el "algoritmo genético". Se utiliza la libreria `pygad`


In [3]:
def fidelidad(J, delta):
    '''
    Se calcula la fidelidad de transmisión a partir de el vector de acoplamientos J.
    Para ello, se crea el hamiltoniano H, luego se diagonaliza y a partir de esto se
    calcula la proyeccion del estado inicial sobre el estado en el que la transmisión
    fue realizada. Para más información del problema físico, consultar: 
    https://doi.org/10.1007/978-3-642-39937-4
    '''

    n = 2*J.size+1
    nj = 2*J.size
    H = np.full((n, n), 0.)

    JJ = np.zeros(nj)

    for i in range(0, J.size, 1):
        JJ[i] = J[i]
        JJ[nj-i-1] = J[i]

    sumj = -0.25*np.sum(JJ)

    for i in range(0, n):
        if (i == 0):
            H[i, i] = sumj+0.5*JJ[i]
        if (i == n-1):
            H[i, i] = sumj + 0.5*JJ[i-1]
        else:
            H[i, i] = sumj + 0.5*JJ[i] + 0.5*JJ[i-1]

        H[i, i] = H[i, i]*delta

    for i in range(0, n-1):
        H[i, i+1] = JJ[i]*(-0.5)
        H[i+1, i] = H[i, i+1]

  # Diagonalizamos este hamiltoniano

    nrows, ncol = H.shape
    (eigvals, eigvects) = la.eig(H)

    c1cn = np.full(ncol, 0.)

    for i in range(0, c1cn.size):
        c1cn[i] = eigvects[0, i]*eigvects[nrows-1, i]

    F = 0.
    Fr = 0.
    Fi = 0.

    t = n

    for i in range(0, nrows):
        Fr = Fr + math.cos(eigvals[i]*t)*c1cn[i]
        Fi = Fi + math.sin(eigvals[i]*t)*c1cn[i]

    Fr = np.real(Fr)
    Fi = np.real(Fi)

    F = Fr*Fr+Fi*Fi

    return F


In [4]:
def fitness_func(solution, solution_idx):
    f = fidelidad(solution, delta)
    fitness = f
    return fitness


In [5]:
def generate_gsp(nj, maxj):
    '''
    Se genera el espacio de genes donde el mínimo es 0 y el máximo puede elegirse para limitar 
    el rango de posibilidades.
    '''
    genespace = []

    for i in range(0, nj//2):
        genespace = np.append(genespace, {'low': 0, 'high': maxj})

    return genespace


Se genera un archivo para obtener una curva de contorno F(nj,delta) para comparar la fidelidad obtenida para distintas dimensiones y para distinto peso del modelo Hxx o Heisenberg. 

In [ ]:
with open("fvsdelta.dat", "w") as f1:
    writer = csv.writer(f1,  delimiter=' ')

    for nj in range(16, 40):

        delta = 0.
        genespace = generate_gsp(nj, nj)
        print(nj)
        writer.writerow(' ')
        id = 0
        while delta <= 1:
            id = id + 1
            delta = 0.05*id

            ga_instance = pygad.GA(num_generations=5000,
                                   num_parents_mating=50,
                                   fitness_func=fitness_func,
                                   sol_per_pop=500,
                                   num_genes=nj//2,
                                   init_range_low=1,
                                   init_range_high=nj,
                                   parent_selection_type='sss',
                                   keep_parents=1,
                                   crossover_type='uniform',
                                   crossover_probability=0.5,
                                   gene_space=genespace,
                                   mutation_type='random',
                                   random_mutation_min_val=-1,
                                   random_mutation_max_val=1,
                                   mutation_probability=0.5,
                                   stop_criteria=['saturate_100'])

            ga_instance.run()
            solution, solution_fitness, solution_idx = ga_instance.best_solution()
            row = [nj, '{:.8f}'.format(delta), '{:.8f}'.format(
                fidelidad(solution, delta))]
            writer.writerow(row)


In [ ]:
with open("fvsdeltan15.dat", "w") as f1:

    writer = csv.writer(f1,  delimiter=' ')
    nj = 15
    delta = 0.
    genespace = generate_gsp(nj, nj)
    print(nj)
    writer.writerow('     ')
    id = 0
    while delta <= 1:
        id = id + 1
        delta = 0.01*id

        ga_instance = pygad.GA(num_generations=5000,
                               num_parents_mating=50,
                               fitness_func=fitness_func,
                               sol_per_pop=500,
                               num_genes=nj//2,
                               init_range_low=1,
                               init_range_high=nj,
                               parent_selection_type='sss',
                               keep_parents=1,
                               crossover_type='uniform',
                               crossover_probability=0.5,
                               gene_space=genespace,
                               mutation_type='random',
                               random_mutation_min_val=-1,
                               random_mutation_max_val=1,
                               mutation_probability=0.5,
                               stop_criteria=['saturate_100'])

        ga_instance.run()
        solution, solution_fitness, solution_idx = ga_instance.best_solution()
        row = [nj, '{:.8f}'.format(delta), '{:.8f}'.format(
            fidelidad(solution, delta))]
        writer.writerow(row)


In [ ]:
with open("fvsdeltan40.dat", "w") as f1:

    writer = csv.writer(f1,  delimiter=' ')
    nj = 40
    delta = 0.
    genespace = generate_gsp(nj, nj)
    print(nj)
    writer.writerow('     ')
    id = 0
    while delta <= 1:
        id = id + 1
        delta = 0.01*id

        ga_instance = pygad.GA(num_generations=5000,
                               num_parents_mating=50,
                               fitness_func=fitness_func,
                               sol_per_pop=500,
                               num_genes=nj//2,
                               init_range_low=1,
                               init_range_high=nj,
                               parent_selection_type='sss',
                               keep_parents=1,
                               crossover_type='uniform',
                               crossover_probability=0.5,
                               gene_space=genespace,
                               mutation_type='random',
                               random_mutation_min_val=-1,
                               random_mutation_max_val=1,
                               mutation_probability=0.5,
                               stop_criteria=['saturate_100'])

        ga_instance.run()
        solution, solution_fitness, solution_idx = ga_instance.best_solution()
        row = [nj, '{:.8f}'.format(delta), '{:.8f}'.format(
            fidelidad(solution, delta))]
        writer.writerow(row)
